## Analysis of Results

#### Setup

In [1]:
### Imports ###
import os
import shutil
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.lines as mlines
import numpy as np
from matplotlib.patches import Patch
from pathlib import Path

In [2]:
work_dir = "/jumbo/keller-lab/Jeremy_Wang/eplus_sa/scripts/main" # Change this to your working directory
os.chdir(work_dir) # Set working directory

In [ ]:
# --- Load Simulation Results for all seeds ---

# number of sims per seed
num_simulations = 30000

# list your seed directories
seeds = [f"seed_{i}" for i in range(1,6)]

# where your outputs live
output_base = os.path.join(work_dir, "output_sobol")

def read_csv_for_sim(task):
    seed, sim_id = task
    folder   = f"randomized_{sim_id}" # sim id is number run within the seed
    csv_file = os.path.join(output_base, seed, folder, "eplusmtr.csv")
    # skip if missing or zero‐byte
    if not os.path.isfile(csv_file) or os.path.getsize(csv_file) == 0:
        return None
    try:
        df = pd.read_csv(csv_file)
    except pd.errors.EmptyDataError:
        return None
    df["Simulation_ID"] = sim_id
    df["seed"]          = seed
    return df

# build the full list of (seed, sim_id) pairs
tasks = [(seed, i) for seed in seeds for i in range(1, num_simulations+1)]

all_dfs = []
max_workers = os.cpu_count() or 4 # returns the number of available cpu cores
print(f"Reading {len(tasks)} files across {len(seeds)} seeds using {max_workers} workers...")

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    future_to_task = {executor.submit(read_csv_for_sim, t): t for t in tasks} # for every task (seed, i) combo, tells the executor to read csv
    for future in as_completed(future_to_task): # as soon as completed, append df into a bigger output
        df = future.result()
        seed, sim_id = future_to_task[future]
        if df is not None:
            all_dfs.append(df)
        else:
            # you can comment this out if it's too noisy
            print(f"Skipping empty/missing file: {seed}/randomized_{sim_id}/eplusmtr.csv")

if not all_dfs:
    raise RuntimeError("No simulation CSV files found. Check your output directories.")

# combine into one big DataFrame
# # rows = ((num_simulations * num_seeds) - (missing files)) * 12 months
combined_df = pd.concat(all_dfs, ignore_index=True)

Reading 150000 files across 5 seeds using 64 workers...
Skipping empty/missing file: seed_1/randomized_3214/eplusmtr.csv
Skipping empty/missing file: seed_3/randomized_744/eplusmtr.csv
Skipping empty/missing file: seed_3/randomized_7061/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_3899/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_3906/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_3908/eplusmtr.csv
Skipping empty/missing file: seed_5/randomized_4294/eplusmtr.csv


In [ ]:
analysis_sim_dir = os.path.join(work_dir, "analysis_sobol")
os.makedirs(analysis_sim_dir, exist_ok=True)

# --- Clean analysis output directory ---
for fn in os.listdir(analysis_sim_dir):
    path = os.path.join(analysis_sim_dir, fn)
    if os.path.isfile(path) or os.path.islink(path): 
        os.unlink(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)

In [ ]:
### Convert numbers into numerics ###


In [ ]:
### Calculate annual energy consumption ###
# note: the column electricity: facility [J] is total electricity consumption in Joules for the month



In [28]:
combined_df

,Date/Time,Electricity:Facility [J](Monthly),Electricity:Building [J](Monthly),InteriorLights:Electricity [J](Monthly),Electricity:HVAC [J](Monthly),NaturalGas:Facility [J](Monthly),NaturalGas:HVAC [J](Monthly),Electricity:Facility [J](RunPeriod),Electricity:Building [J](RunPeriod),InteriorLights:Electricity [J](RunPeriod),Electricity:HVAC [J](RunPeriod),NaturalGas:Facility [J](RunPeriod),NaturalGas:HVAC [J](RunPeriod),Simulation_ID,seed
0,January,1.676559e+10,7.076032e+09,9.379652e+08,9.689554e+09,1.311726e+11,1.311726e+11,NaN,NaN,NaN,NaN,NaN,NaN,41,seed_1
1,February,1.581216e+10,7.025878e+09,8.519067e+08,8.786286e+09,1.161166e+11,1.161166e+11,NaN,NaN,NaN,NaN,NaN,NaN,41,seed_1
2,March,1.842470e+10,8.515903e+09,9.744858e+08,9.908797e+09,1.245478e+11,1.245478e+11,NaN,NaN,NaN,NaN,NaN,NaN,41,seed_1
3,April,1.844575e+10,9.754929e+09,8.605849e+08,8.690816e+09,9.096869e+10,9.096869e+10,NaN,NaN,NaN,NaN,NaN,NaN,41,seed_1
4,May,1.798981e+10,1.049796e+10,9.744858e+08,7.491846e+09,6.004582e+10,6.004582e+10,NaN,NaN,NaN,NaN,NaN,NaN,41,seed_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1799911,August,9.377652e+09,6.409374e+09,9.292329e+08,2.968278e+09,5.679528e+09,5.679528e+09,NaN,NaN,NaN,NaN,NaN,NaN,29897,seed_5
1799912,September,1.318761e+10,8.407485e+09,8.554460e+08,4.780127e+09,2.017084e+10,2.017084e+10,NaN,NaN,NaN,NaN,NaN,NaN,29897,seed_5
1799913,October,1.890872e+10,1.066484e+10,8.944083e+08,8.243879e+09,7.297701e+10,7.297701e+10,NaN,NaN,NaN,NaN,NaN,NaN,29897,seed_5
1799914,November,1.887586e+10,9.310729e+09,8.902707e+08,9.565128e+09,1.093404e+11,1.093404e+11,NaN,NaN,NaN,NaN,NaN,NaN,29897,seed_5
